# CXR-LT 2024

This notebook mirrors the CXR-LT 2023 examination pass for the 2024 release. It uses `labels.csv` for the combined labeled view and keeps task-specific files available for schema and task checks.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from utils import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 140)

root_dir, data_dir = get_notebook_paths()
cxr_lt_2024_dir = data_dir / "CXR-LT" / "cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0" / "cxr-lt-2024"
mimic_cxr_dir = data_dir / "MIMIC-CXR"
mimic_cxr_jpg_dir = data_dir / "MIMIC-CXR-JPG"

print(f"root_dir: {root_dir}")
print(f"cxr_lt_2024_dir: {cxr_lt_2024_dir}")
print(f"mimic_cxr_dir exists: {mimic_cxr_dir.exists()}")
print(f"mimic_cxr_jpg_dir exists: {mimic_cxr_jpg_dir.exists()}")


In [ ]:
CSV_FILES = {
    "labels": "labels.csv",
    "train": "train_labeled.csv",
    "development_task1": "development_labeled_task1.csv",
    "development_task2": "development_labeled_task2.csv",
    "development_task3": "development_labeled_task3.csv",
    "test_task1": "test_labeled_task1.csv",
    "test_task2": "test_labeled_task2.csv",
    "test_task3": "test_labeled_task3.csv",
}

datasets = load_csv_map(cxr_lt_2024_dir, CSV_FILES)
global_df = datasets["labels"].copy()
label_cols = label_columns(global_df)
analysis_splits = {split_name: df.copy() for split_name, df in global_df.groupby("split", sort=False)}
global_frames = {"global": global_df}
task_splits = {name: df for name, df in datasets.items() if name != "labels"}

print(f"Loaded {len(datasets)} files")
print(f"Detected {len(label_cols)} labels")
print(f"Global analysis rows: {len(global_df):,}")
print(f"Path column: {detect_path_column(global_df)}")
label_cols


## 1) Quick File And Split Overview


In [ ]:
for name, df in datasets.items():
    preview_dataset(name, df)

split_overview_df = build_split_overview(analysis_splits, label_cols)
display(split_overview_df)


## 2) Task File Schema Checks


In [ ]:
task_schema_df = pd.DataFrame(
    [
        {
            "file": name,
            "rows": len(df),
            "columns": len(df.columns),
            "labels": len(label_columns(df)),
            "path_column": detect_path_column(df),
            "subjects": df["subject_id"].nunique(),
            "studies": df["study_id"].nunique(),
            "dicoms": df["dicom_id"].nunique(),
        }
        for name, df in task_splits.items()
    ]
)
display(task_schema_df)

task_label_sets = {name: set(label_columns(df)) for name, df in task_splits.items()}
label_membership_df = pd.DataFrame(
    [
        {"label": label, **{name: label in labels for name, labels in task_label_sets.items()}}
        for label in sorted(set().union(*task_label_sets.values()))
    ]
)
display(label_membership_df)


## 3) Global Dataset Statistics


In [ ]:
global_overview_df = build_split_overview(global_frames, label_cols).rename(columns={"split": "scope"})
global_label_summary_df = summarize_labels(global_df, "global", label_cols)
global_view_position_summary_df = summarize_view_positions(global_df, "global", top_n=12).rename(columns={"split": "scope"})
global_normal_conflicts_df = summarize_no_finding_conflicts(global_frames, label_cols).rename(columns={"split": "scope"})

display(global_overview_df)
display(global_label_summary_df)
display(global_view_position_summary_df)
display(global_normal_conflicts_df)

plot_global_label_prevalence(global_df, label_cols, top_n=min(40, len(label_cols)))
plot_global_view_positions(global_df, top_n=12)


## 4) Label Imbalance And Multi-Label Density


In [ ]:
label_summary_df = pd.concat(
    [summarize_labels(df, split_name, label_cols) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
label_summary_with_global_df = pd.concat([label_summary_df, global_label_summary_df], ignore_index=True)

display(label_summary_with_global_df)
plot_label_prevalence(analysis_splits, label_cols, global_df, top_n=20)
plot_labels_per_image(analysis_splits, label_cols)


## 5) View Metadata And Split Leakage Checks


In [ ]:
view_position_summary_df = pd.concat(
    [summarize_view_positions(df, split_name, top_n=999) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
view_position_summary_with_global_df = pd.concat([view_position_summary_df, global_view_position_summary_df], ignore_index=True)

display(view_position_summary_with_global_df)
plot_view_positions(analysis_splits, global_df, top_n=8)

print("Subject overlap across splits")
display(compute_overlap(analysis_splits, "subject_id"))
print("Study overlap across splits")
display(compute_overlap(analysis_splits, "study_id"))
print("DICOM overlap across splits")
display(compute_overlap(analysis_splits, "dicom_id"))


## 6) Study-Level Aggregation


In [ ]:
study_table_df, study_view_summary_df = summarize_study_views(analysis_splits, label_cols)
global_study_table_df, global_study_view_summary_df = summarize_study_views(global_frames, label_cols)
global_study_view_summary_df = global_study_view_summary_df.rename(columns={"split": "scope"})

display(study_view_summary_df)
display(global_study_view_summary_df)

display(summarize_view_combos(study_table_df).sort_values(["split", "study_count"], ascending=[True, False]).head(30))
display(summarize_view_combos(global_study_table_df).sort_values("study_count", ascending=False).head(20))

plot_top_view_combos(study_table_df, top_n=12)
plot_global_top_view_combos(global_study_table_df, top_n=12)


## 7) Patient Timeline Behavior


In [ ]:
metadata_df, metadata_path = load_first_existing(
    [
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv.gz",
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv",
    ]
)

if metadata_df is None:
    print("MIMIC-CXR-JPG metadata was not found. Timeline cells will fall back to study_id ordering.")
    metadata_for_join = pd.DataFrame(columns=["dicom_id", "study_datetime"])
else:
    print(f"Loaded metadata: {metadata_path}")
    metadata_df["study_datetime"] = parse_mimic_study_datetime(metadata_df)
    metadata_columns = [
        column
        for column in [
            "dicom_id",
            "StudyDate",
            "StudyTime",
            "study_datetime",
            "PerformedProcedureStepDescription",
            "ProcedureCodeSequence_CodeMeaning",
            "Rows",
            "Columns",
        ]
        if column in metadata_df.columns
    ]
    metadata_for_join = metadata_df[metadata_columns].copy()

all_labeled_images_df = global_df.merge(metadata_for_join, on="dicom_id", how="left")
print(f"Rows with parsed study_datetime: {all_labeled_images_df['study_datetime'].notna().sum():,} / {len(all_labeled_images_df):,}")
display(all_labeled_images_df.head(3))


In [ ]:
study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols, scope_columns=["split"])
global_study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols)

patient_timeline_summary_df, split_patient_summary_df = summarize_patient_timelines(study_timeline_df, scope_columns=["split"])
global_patient_timeline_summary_df, global_patient_summary_df = summarize_patient_timelines(global_study_timeline_df)

display(split_patient_summary_df)
display(global_patient_summary_df)
display(global_patient_timeline_summary_df.sort_values(["study_count", "span_days"], ascending=[False, False]).head(10))


## 8) Global Label Co-Occurrence And Correlations


In [ ]:
global_corr_df = global_df[label_cols].corr()
pair_corr_df = label_pair_correlation_table(global_df, label_cols)

plt.figure(figsize=(16, 14))
sns.heatmap(global_corr_df, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, linewidths=0.2)
plt.title("CXR-LT 2024 global label correlation matrix")
plt.tight_layout()
plt.show()

print("Most positively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=False).head(20))

print("Most negatively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=True).head(20))


In [ ]:
global_cooccurrence_df = global_df[label_cols].T.dot(global_df[label_cols])
label_order = global_df[label_cols].sum().sort_values(ascending=False).index

plt.figure(figsize=(16, 14))
sns.heatmap(global_cooccurrence_df.loc[label_order, label_order], cmap="mako", square=True, linewidths=0.2)
plt.title("CXR-LT 2024 global label co-occurrence counts")
plt.tight_layout()
plt.show()

display(global_cooccurrence_df.loc[label_order, label_order].iloc[:15, :15])


## 9) View-Conditioned Prevalence


In [ ]:
view_prevalence_df = view_conditioned_prevalence(analysis_splits, label_cols)
global_view_prevalence_df = view_conditioned_prevalence(global_frames, label_cols)

display(view_prevalence_df[["split", "ViewPosition", "row_count"]].drop_duplicates().sort_values(["split", "ViewPosition"]))

top_global_labels = global_df[label_cols].sum().sort_values(ascending=False).head(15).index.tolist()
plot_view_conditioned_prevalence(global_view_prevalence_df, label_order=top_global_labels, title="CXR-LT 2024 global label prevalence by view position")

display(
    global_view_prevalence_df
    .query("label in @top_global_labels")
    .pivot(index="label", columns="ViewPosition", values="positive_rate_pct")
    .loc[top_global_labels]
    .round(3)
)


## 10) Image Availability And Path Sanity


In [ ]:
image_root = mimic_cxr_jpg_dir
print(f"image_root: {image_root}")
print(f"image_root exists: {image_root.exists()}")
print(f"image files directory exists: {(image_root / 'files').exists()}")

image_path_summary_df = summarize_image_paths(analysis_splits, image_root)
global_image_path_summary_df = summarize_image_paths(global_frames, image_root).rename(columns={"split": "scope"})

display(image_path_summary_df)
display(global_image_path_summary_df)

path_column = detect_path_column(global_df)
missing_global_paths = global_df.loc[
    ~global_df[path_column].astype(str).map(lambda value: resolve_relative_path(image_root, value).is_file()),
    ["split", "dicom_id", "subject_id", "study_id", "ViewPosition", path_column],
].head(10)

print("Sample missing global image paths")
display(missing_global_paths)


## 11) Text Linkage Readiness


In [ ]:
study_list_df, study_list_path = load_first_existing(
    [
        mimic_cxr_dir / "cxr-study-list.csv.gz",
        mimic_cxr_dir / "cxr-study-list.csv",
    ]
)

if study_list_df is None:
    print("MIMIC-CXR study list was not found. Report linkage cannot be checked.")
    report_link_summary_df = pd.DataFrame()
    global_report_link_summary_df = pd.DataFrame()
    linked_reports_df = pd.DataFrame()
    global_linked_reports_df = pd.DataFrame()
else:
    print(f"Loaded study list: {study_list_path}")
    (
        report_link_summary_df,
        global_report_link_summary_df,
        linked_reports_df,
        global_linked_reports_df,
    ) = build_report_linkage(analysis_splits, global_df, study_list_df, mimic_cxr_dir)

    print(f"report files directory exists: {(mimic_cxr_dir / 'files').exists()}")
    display(report_link_summary_df)
    display(global_report_link_summary_df)
    display(linked_reports_df.head(10))


In [ ]:
if global_linked_reports_df.empty or not global_linked_reports_df["report_file_exists"].any():
    print("No local report text files were found to preview.")
else:
    sample_report_row = global_linked_reports_df.query("report_file_exists").iloc[0]
    print(
        "Sample report: "
        f"subject_id={sample_report_row['subject_id']}, "
        f"study_id={sample_report_row['study_id']}"
    )
    print(read_report_text(mimic_cxr_dir, sample_report_row["report_path"]))


## Notes For The Next Modeling Pass

Use these outputs to decide whether the active dataloader should target the 2023 label set, the larger 2024 task-1 label set, or a shared-label compatibility subset.
